# 04h — Siamese Euclidean V3: Tumor + Healthy

## Obiettivo

Addestrare nuovamente la Siamese Network Euclidean V3 utilizzando
esclusivamente il task Tumor + Healthy.

Il precedente encoder Euclidean V3 era stato addestrato su 18 classi,
comprendenti anche patologie non oncologiche.

L'audit del dataset ha mostrato tre gruppi distinti:

- cancer
- healthy
- non_cancer_disease

Poiché l'obiettivo del progetto è associare un sample eccDNA al tipo
di tumore includendo anche Healthy, questo esperimento utilizza solamente
le 11 classi oncologiche e la classe Healthy.

La rete viene addestrata da zero per ottenere una baseline corretta
del task a 12 classi, senza ereditare la rappresentazione appresa
sul precedente problema a 18 classi.

## Configurazione

- FCGR k=6
- 12 classi Tumor + Healthy
- CNN V3 bias-free
- embedding 128D L2-normalizzato
- Euclidean Contrastive Loss
- margin iniziale = 1.25
- positive / negative pairs ≈ 50/50
- split basato su `split_cluster`
- training bilanciato tramite downsampling
- checkpoint selezionato tramite validation ROC-AUC pairwise
- successiva valutazione multiclass per similarità

In [1]:
from pathlib import Path

import json
import random
import time
import copy
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)


print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.12.0+cu126
CUDA: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# ============================================================
# CELL 3 — PATH
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "siamese_euclidean_v3_tumor_healthy"
)

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_manifest.tsv"
)


VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


PAIR_CONFIG_PATH = (
    PROCESSED_DIR
    / "siamese_pair_config.json"
)


with open(
    PAIR_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    pair_config = json.load(f)


K = int(
    pair_config["k"]
)

RANDOM_STATE = int(
    pair_config["random_state"]
)


FCGR_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}.npy"
)


FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}_index.tsv"
)


EMBEDDING_DIM = 128
BATCH_SIZE = 128

TRAIN_PAIRS_PER_EPOCH = int(
    pair_config["train_pairs_per_epoch"]
)

VAL_PAIRS = int(
    pair_config["val_pairs"]
)

POSITIVE_PROBABILITY = 0.5


print("k:", K)
print("Batch:", BATCH_SIZE)
print("Train pairs/epoch:", TRAIN_PAIRS_PER_EPOCH)
print("Val pairs:", VAL_PAIRS)

k: 6
Batch: 128
Train pairs/epoch: 50000
Val pairs: 10000


In [3]:
# ============================================================
# CELL 4 — SEED + DEVICE
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(
    RANDOM_STATE
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )


print(
    "Device:",
    DEVICE
)

Device: cuda


In [4]:
# ============================================================
# CELL 5 — TUMOR + HEALTHY DATA
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={"id": str}
)


metadata["class_id"] = (
    metadata["class_id"]
    .astype(int)
)


train_full = (
    metadata[
        metadata["split_cluster"]
        ==
        "train"
    ]
    .copy()
)


# ============================================================
# FIXED DOWNSAMPLING
# ============================================================

min_train_class_size = int(
    train_full[
        "class_id"
    ]
    .value_counts()
    .min()
)


balanced_parts = []


for class_id, group in train_full.groupby(
    "class_id"
):

    selected = group.sample(
        n=min_train_class_size,
        replace=False,
        random_state=
            RANDOM_STATE
            +
            int(class_id)
    )

    balanced_parts.append(
        selected
    )


train_metadata = (
    pd.concat(
        balanced_parts,
        ignore_index=True
    )
)


print(
    "Classe minima:",
    min_train_class_size
)

print(
    "Train bilanciato:",
    len(train_metadata)
)

print(
    "Numero classi:",
    train_metadata[
        "class_id"
    ].nunique()
)


display(
    train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("n_train")
    .to_frame()
)

Classe minima: 2111
Train bilanciato: 25332
Numero classi: 12


,n_train
class_id,
0,2111
1,2111
2,2111
3,2111
4,2111
5,2111
6,2111
7,2111
8,2111


In [5]:
# ============================================================
# CELL 6 — FILTER + REMAP VALIDATION
# ============================================================

CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_class_mapping.tsv"
)


class_mapping = pd.read_csv(
    CLASS_MAPPING_PATH,
    sep="\t"
)


old_to_new = dict(
    zip(
        class_mapping[
            "original_class_id"
        ].astype(int),

        class_mapping[
            "class_id"
        ].astype(int)
    )
)


val_pool_original = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={"id": str}
)


val_pool_original[
    "class_id"
] = (
    val_pool_original[
        "class_id"
    ]
    .astype(int)
)


included_ids = set(
    old_to_new.keys()
)


val_metadata = (
    val_pool_original[
        val_pool_original[
            "class_id"
        ].isin(
            included_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


val_metadata[
    "original_class_id"
] = (
    val_metadata[
        "class_id"
    ]
)


val_metadata[
    "class_id"
] = (

    val_metadata[
        "original_class_id"
    ]
    .map(
        old_to_new
    )
    .astype(int)
)


print(
    "Validation samples:",
    len(val_metadata)
)

print(
    "Validation classes:",
    val_metadata[
        "class_id"
    ].nunique()
)


display(
    val_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("n_val")
    .to_frame()
)

Validation samples: 9753
Validation classes: 12


,n_val
class_id,
0,1000
1,1000
2,1000
3,1000
4,1000
5,1000
6,1000
7,1000
8,585


In [6]:
# ============================================================
# CELL 7 — FCGR
# ============================================================

fcgr_memmap = np.load(
    FCGR_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={"id": str}
)


id_to_fcgr_row = dict(
    zip(
        fcgr_index["id"],
        fcgr_index["fcgr_row"]
    )
)


missing_train = (
    ~train_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


missing_val = (
    ~val_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


print(
    "FCGR:",
    fcgr_memmap.shape,
    fcgr_memmap.dtype
)

print(
    "Missing train:",
    missing_train
)

print(
    "Missing val:",
    missing_val
)


assert missing_train == 0
assert missing_val == 0

FCGR: (150272, 64, 64) float32
Missing train: 0
Missing val: 0


In [7]:
# ============================================================
# CELL 8 — OPTIMIZED SIAMESE PAIR DATASET
# ============================================================

class SiamesePairDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row,
        n_pairs,
        positive_fraction=0.5,
        seed=42,
        dynamic=False
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap

        self.n_pairs = int(n_pairs)

        self.positive_fraction = float(
            positive_fraction
        )

        self.seed = int(seed)

        self.dynamic = bool(dynamic)


        # ----------------------------------------------------
        # Pre-compute FCGR rows
        # ----------------------------------------------------

        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


        self.classes = np.array(
            sorted(
                np.unique(
                    self.labels
                )
            ),
            dtype=np.int64
        )


        self.class_to_indices = {

            int(class_id):
                np.where(
                    self.labels == class_id
                )[0]

            for class_id
            in self.classes
        }


        self.epoch = 0


        # Pair iniziali
        self._generate_pairs(
            self.seed
        )


    def _generate_pairs(
        self,
        seed
    ):

        rng = np.random.default_rng(
            seed
        )


        n_positive = int(
            round(
                self.n_pairs
                *
                self.positive_fraction
            )
        )


        targets = np.zeros(
            self.n_pairs,
            dtype=np.float32
        )


        targets[
            :n_positive
        ] = 1.0


        rng.shuffle(
            targets
        )


        anchors = rng.integers(
            low=0,
            high=len(self.labels),
            size=self.n_pairs
        )


        partners = np.empty(
            self.n_pairs,
            dtype=np.int64
        )


        for i in range(
            self.n_pairs
        ):

            anchor_idx = int(
                anchors[i]
            )

            anchor_class = int(
                self.labels[
                    anchor_idx
                ]
            )


            # =================================================
            # POSITIVE PAIR
            # =================================================

            if targets[i] == 1.0:

                candidates = (
                    self.class_to_indices[
                        anchor_class
                    ]
                )


                partner_idx = anchor_idx


                while (
                    partner_idx
                    ==
                    anchor_idx
                ):

                    partner_idx = int(
                        rng.choice(
                            candidates
                        )
                    )


            # =================================================
            # NEGATIVE PAIR
            # =================================================

            else:

                negative_classes = (
                    self.classes[
                        self.classes
                        !=
                        anchor_class
                    ]
                )


                negative_class = int(
                    rng.choice(
                        negative_classes
                    )
                )


                partner_idx = int(
                    rng.choice(
                        self.class_to_indices[
                            negative_class
                        ]
                    )
                )


            partners[i] = (
                partner_idx
            )


        self.row1 = (
            self.rows[
                anchors
            ]
        )

        self.row2 = (
            self.rows[
                partners
            ]
        )

        self.targets = (
            targets
        )


    def set_epoch(
        self,
        epoch
    ):

        self.epoch = int(
            epoch
        )


        if self.dynamic:

            self._generate_pairs(

                self.seed
                +
                self.epoch
                *
                100_003
            )


    def __len__(
        self
    ):

        return (
            self.n_pairs
        )


    def __getitem__(
        self,
        index
    ):

        row1 = int(
            self.row1[index]
        )

        row2 = int(
            self.row2[index]
        )


        fcgr1 = np.array(
            self.fcgr_memmap[
                row1
            ],
            dtype=np.float32,
            copy=True
        )


        fcgr2 = np.array(
            self.fcgr_memmap[
                row2
            ],
            dtype=np.float32,
            copy=True
        )


        return {

            "x1":
                torch.from_numpy(
                    fcgr1
                ).unsqueeze(0),

            "x2":
                torch.from_numpy(
                    fcgr2
                ).unsqueeze(0),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.float32
                )
        }

In [8]:
# ============================================================
# CELL 9 — PAIR LOADERS
# ============================================================

train_pair_dataset = SiamesePairDataset(

    metadata=
        train_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row,

    n_pairs=
        TRAIN_PAIRS_PER_EPOCH,

    positive_fraction=
        POSITIVE_PROBABILITY,

    seed=
        RANDOM_STATE,

    dynamic=True
)


val_pair_dataset = SiamesePairDataset(

    metadata=
        val_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row,

    n_pairs=
        VAL_PAIRS,

    positive_fraction=
        0.5,

    seed=
        RANDOM_STATE
        +
        50_000,

    dynamic=False
)


train_pair_loader = DataLoader(

    train_pair_dataset,

    batch_size=
        BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


val_pair_loader = DataLoader(

    val_pair_dataset,

    batch_size=
        BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print(
    "Train pairs:",
    len(train_pair_dataset)
)

print(
    "Validation pairs:",
    len(val_pair_dataset)
)


print(
    "Train positive fraction:",
    train_pair_dataset.targets.mean()
)

print(
    "Val positive fraction:",
    val_pair_dataset.targets.mean()
)


batch_check = next(
    iter(
        train_pair_loader
    )
)


print()
print(
    "x1:",
    batch_check["x1"].shape
)

print(
    "x2:",
    batch_check["x2"].shape
)

print(
    "target:",
    batch_check["target"].shape
)

Train pairs: 50000
Validation pairs: 10000
Train positive fraction: 0.5
Val positive fraction: 0.5

x1: torch.Size([128, 1, 64, 64])
x2: torch.Size([128, 1, 64, 64])
target: torch.Size([128])


In [9]:
# ============================================================
# CELL 10 — EUCLIDEAN V3
# ============================================================

class FCGRCNNEncoderV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                1, 32, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                32
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                32, 32, 3,
                padding=1,
                bias=False
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                32, 64, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                64, 128, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                128, 128, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.AdaptiveAvgPool2d(
                (4, 4)
            )
        )


        self.embedding_head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                256,
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        x = self.features(
            x
        )


        z = self.embedding_head(
            x
        )


        return F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8
        )


class SiameseNetworkEuclideanV3(
    nn.Module
):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()

        self.encoder = (
            FCGRCNNEncoderV3(
                embedding_dim
            )
        )


    def forward(
        self,
        x1,
        x2
    ):

        batch_size = (
            x1.shape[0]
        )


        # Un solo forward CNN
        x = torch.cat(
            [
                x1,
                x2
            ],
            dim=0
        )


        z = self.encoder(
            x
        )


        z1 = z[
            :batch_size
        ]

        z2 = z[
            batch_size:
        ]


        return (
            z1,
            z2
        )


model = (
    SiameseNetworkEuclideanV3(
        embedding_dim=
            EMBEDDING_DIM
    )
    .to(
        DEVICE
    )
)


print(model)

print()

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

SiameseNetworkEuclideanV3(
  (encoder): FCGRCNNEncoderV3(
    (features): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): GroupNorm(8, 32, eps=1e-05, affine=True, bias=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): ReLU(inplace=True)
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (7): GroupNorm(8, 64, eps=1e-05, affine=True, bias=True)
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (11): GroupNorm(8, 128, eps=1e-05, affine=True, bias=True)
      (12): ReLU(inplace=True)
      (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, cei

In [10]:
# ============================================================
# CELL 11 — EUCLIDEAN CONTRASTIVE LOSS
# ============================================================

EUCLIDEAN_MARGIN = 1.25


class EuclideanContrastiveLoss(
    nn.Module
):

    def __init__(
        self,
        margin=1.25
    ):

        super().__init__()

        self.margin = float(
            margin
        )


    def forward(
        self,
        z1,
        z2,
        target
    ):

        z1 = z1.float()
        z2 = z2.float()

        target = target.float()


        distances = torch.linalg.vector_norm(
            z1 - z2,
            ord=2,
            dim=1
        )


        positive_loss = (
            target
            *
            distances.pow(2)
        )


        negative_loss = (

            (1.0 - target)

            *

            F.relu(
                self.margin
                -
                distances
            ).pow(2)
        )


        loss = (
            positive_loss
            +
            negative_loss
        ).mean()


        return (
            loss,
            distances
        )


criterion = (
    EuclideanContrastiveLoss(
        margin=
            EUCLIDEAN_MARGIN
    )
)


print(
    "Margin:",
    criterion.margin
)

Margin: 1.25


In [12]:
# ============================================================
# CELL 12 — PAIRWISE EVALUATION
# ============================================================

def evaluate_pairwise(
    model,
    loader,
    criterion
):

    model.eval()


    total_loss = 0.0
    total_samples = 0


    all_targets = []
    all_distances = []


    with torch.no_grad():

        for batch in loader:

            x1 = (
                batch["x1"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            x2 = (
                batch["x2"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            target = (
                batch["target"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(

                device_type=
                    DEVICE.type,

                dtype=
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16,

                enabled=
                    DEVICE.type == "cuda"

            ):

                z1, z2 = model(
                    x1,
                    x2
                )


            (
                loss,
                distances
            ) = criterion(
                z1,
                z2,
                target
            )


            batch_size = (
                target.shape[0]
            )


            total_loss += (
                loss.item()
                *
                batch_size
            )

            total_samples += (
                batch_size
            )


            all_targets.append(
                target
                .cpu()
                .numpy()
            )


            all_distances.append(
                distances
                .cpu()
                .numpy()
            )


    targets = np.concatenate(
        all_targets
    )

    distances = np.concatenate(
        all_distances
    )


    positive_distances = (
        distances[
            targets == 1
        ]
    )

    negative_distances = (
        distances[
            targets == 0
        ]
    )


    d_pos = float(
        positive_distances.mean()
    )

    d_neg = float(
        negative_distances.mean()
    )


    gap = (
        d_neg
        -
        d_pos
    )


    pooled_variance = (

        0.5
        *
        (
            positive_distances.var()
            +
            negative_distances.var()
        )
    )


    d_prime = float(

        gap
        /
        np.sqrt(
            pooled_variance
            +
            1e-12
        )
    )


    auc = float(
        roc_auc_score(
            targets,
            -distances
        )
    )


    return {

        "loss":
            total_loss
            /
            total_samples,

        "auc":
            auc,

        "d_pos":
            d_pos,

        "d_neg":
            d_neg,

        "gap":
            gap,

        "d_prime":
            d_prime
    }

In [13]:
# ============================================================
# CELL 13 — OPTIMIZER + AMP
# ============================================================

LEARNING_RATE = 5e-4

WEIGHT_DECAY = 1e-4


try:

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            LEARNING_RATE,

        weight_decay=
            WEIGHT_DECAY,

        fused=
            DEVICE.type == "cuda"
    )

    fused_adamw = (
        DEVICE.type == "cuda"
    )


except (
    RuntimeError,
    TypeError
):

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            LEARNING_RATE,

        weight_decay=
            WEIGHT_DECAY
    )

    fused_adamw = False


AMP_ENABLED = (
    DEVICE.type
    ==
    "cuda"
)


scaler = torch.amp.GradScaler(

    "cuda",

    enabled=
        AMP_ENABLED
)


print(
    "LR:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "AMP:",
    AMP_ENABLED
)

print(
    "Fused AdamW:",
    fused_adamw
)

LR: 0.0005
Weight decay: 0.0001
AMP: True
Fused AdamW: True


In [14]:
# ============================================================
# CELL 14 — TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    dataset,
    criterion,
    optimizer,
    scaler,
    epoch
):

    model.train()


    # Nuove coppie ogni epoca
    dataset.set_epoch(
        epoch
    )


    total_loss = 0.0
    total_samples = 0


    all_targets = []
    all_distances = []


    start_time = (
        time.perf_counter()
    )


    for batch in loader:

        x1 = (
            batch["x1"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        x2 = (
            batch["x2"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        target = (
            batch["target"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(

            device_type=
                DEVICE.type,

            dtype=
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16,

            enabled=
                AMP_ENABLED

        ):

            z1, z2 = model(
                x1,
                x2
            )


        (
            loss,
            distances
        ) = criterion(
            z1,
            z2,
            target
        )


        if AMP_ENABLED:

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            loss.backward()

            optimizer.step()


        batch_size = (
            target.shape[0]
        )


        total_loss += (
            loss.detach().item()
            *
            batch_size
        )

        total_samples += (
            batch_size
        )


        all_targets.append(
            target
            .detach()
            .cpu()
            .numpy()
        )

        all_distances.append(
            distances
            .detach()
            .cpu()
            .numpy()
        )


    elapsed = (
        time.perf_counter()
        -
        start_time
    )


    targets = np.concatenate(
        all_targets
    )

    distances = np.concatenate(
        all_distances
    )


    train_auc = roc_auc_score(
        targets,
        -distances
    )


    return {

        "loss":
            total_loss
            /
            total_samples,

        "auc":
            float(
                train_auc
            ),

        "seconds":
            elapsed
    }


In [15]:
# ============================================================
# CELL 15 — 3-EPOCH SMOKE TEST
# ============================================================

SMOKE_EPOCHS = 3


initial_model_state = copy.deepcopy(
    model.state_dict()
)


print("=" * 82)
print("EUCLIDEAN V3 — 12-WAY TUMOR + HEALTHY — SMOKE TEST")
print("=" * 82)


for epoch in range(
    1,
    SMOKE_EPOCHS + 1
):

    train_metrics = train_one_epoch(

        model=
            model,

        loader=
            train_pair_loader,

        dataset=
            train_pair_dataset,

        criterion=
            criterion,

        optimizer=
            optimizer,

        scaler=
            scaler,

        epoch=
            epoch
    )


    val_metrics = evaluate_pairwise(

        model=
            model,

        loader=
            val_pair_loader,

        criterion=
            criterion
    )


    print(

        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | train loss "
        f"{train_metrics['loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['auc']:.4f}"

        f" | val loss "
        f"{val_metrics['loss']:.4f}"

        f" | val AUC "
        f"{val_metrics['auc']:.4f}"

        f" | d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

EUCLIDEAN V3 — 12-WAY TUMOR + HEALTHY — SMOKE TEST
Epoch 01/3 | train loss 0.3809 | train AUC 0.6186 | val loss 0.3903 | val AUC 0.6023 | d+ 0.5491 | d- 0.6271 | gap 0.0781 | d' 0.3659 | 34.4s
Epoch 02/3 | train loss 0.3699 | train AUC 0.6391 | val loss 0.3715 | val AUC 0.6439 | d+ 0.5949 | d- 0.7087 | gap 0.1138 | d' 0.5190 | 17.9s
Epoch 03/3 | train loss 0.3669 | train AUC 0.6460 | val loss 0.3702 | val AUC 0.6398 | d+ 0.5859 | d- 0.6858 | gap 0.0999 | d' 0.5035 | 51.1s


In [16]:
# ============================================================
# CELL 16 — RESTORE BEFORE FULL TRAINING
# ============================================================

model.load_state_dict(
    initial_model_state
)


# Nuovo optimizer: nessuno stato ereditato dallo smoke test
try:

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY,

        fused=(DEVICE.type == "cuda")
    )

except (RuntimeError, TypeError):

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY
    )


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


print(
    "Modello ripristinato allo stato iniziale."
)

print(
    "Optimizer reinizializzato."
)

Modello ripristinato allo stato iniziale.
Optimizer reinizializzato.


In [17]:
# ============================================================
# CELL 17 — FULL TRAINING CONFIG
# ============================================================

MAX_EPOCHS = 50

EARLY_STOPPING_PATIENCE = 10

MIN_DELTA = 1e-4


BEST_CHECKPOINT_PATH = (
    ARTIFACTS_DIR
    / "euclidean_v3_tumor_healthy_margin_1p25_best.pt"
)


HISTORY_PATH = (
    ARTIFACTS_DIR
    / "euclidean_v3_tumor_healthy_margin_1p25_history.tsv"
)


print(
    "Max epochs:",
    MAX_EPOCHS
)

print(
    "Patience:",
    EARLY_STOPPING_PATIENCE
)

print(
    "Checkpoint:",
    BEST_CHECKPOINT_PATH
)

Max epochs: 50
Patience: 10
Checkpoint: D:\Daria\Desktop\eccdna_fcgr_siamese\artifacts\siamese_euclidean_v3_tumor_healthy\euclidean_v3_tumor_healthy_margin_1p25_best.pt


In [18]:
# ============================================================
# CELL 18 — FULL TRAINING
# ============================================================

best_val_auc = -np.inf

best_epoch = 0

epochs_without_improvement = 0

history = []


print("=" * 92)
print("EUCLIDEAN V3 — TUMOR + HEALTHY — FULL TRAINING")
print("=" * 92)


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    # ========================================================
    # TRAIN
    # ========================================================

    train_metrics = train_one_epoch(

        model=model,

        loader=train_pair_loader,

        dataset=train_pair_dataset,

        criterion=criterion,

        optimizer=optimizer,

        scaler=scaler,

        epoch=epoch
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    val_metrics = evaluate_pairwise(

        model=model,

        loader=val_pair_loader,

        criterion=criterion
    )


    current_auc = float(
        val_metrics["auc"]
    )


    improved = (
        current_auc
        >
        best_val_auc
        +
        MIN_DELTA
    )


    # ========================================================
    # CHECKPOINT
    # ========================================================

    if improved:

        best_val_auc = current_auc

        best_epoch = epoch

        epochs_without_improvement = 0


        checkpoint = {

            "epoch":
                epoch,

            "best_epoch":
                epoch,

            "best_val_auc":
                current_auc,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "embedding_dim":
                EMBEDDING_DIM,

            "margin":
                EUCLIDEAN_MARGIN,

            "learning_rate":
                LEARNING_RATE,

            "weight_decay":
                WEIGHT_DECAY,

            "k":
                K,

            "n_classes":
                12,

            "task":
                "tumor_healthy",

            "val_loss":
                val_metrics["loss"],

            "d_pos":
                val_metrics["d_pos"],

            "d_neg":
                val_metrics["d_neg"],

            "gap":
                val_metrics["gap"],

            "d_prime":
                val_metrics["d_prime"]
        }


        torch.save(
            checkpoint,
            BEST_CHECKPOINT_PATH
        )


    else:

        epochs_without_improvement += 1


    # ========================================================
    # HISTORY
    # ========================================================

    history.append(
        {

            "epoch":
                epoch,

            "train_loss":
                train_metrics["loss"],

            "train_auc":
                train_metrics["auc"],

            "val_loss":
                val_metrics["loss"],

            "val_auc":
                val_metrics["auc"],

            "d_pos":
                val_metrics["d_pos"],

            "d_neg":
                val_metrics["d_neg"],

            "gap":
                val_metrics["gap"],

            "d_prime":
                val_metrics["d_prime"],

            "seconds":
                train_metrics["seconds"]
        }
    )


    marker = (
        " *BEST*"
        if improved
        else ""
    )


    print(

        f"Epoch {epoch:02d}/{MAX_EPOCHS}"

        f" | train loss "
        f"{train_metrics['loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['auc']:.4f}"

        f" | val loss "
        f"{val_metrics['loss']:.4f}"

        f" | val AUC "
        f"{val_metrics['auc']:.4f}"

        f" | d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"

        f"{marker}"
    )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):

        print()
        print(
            f"Early stopping at epoch {epoch}."
        )

        break


# ============================================================
# SAVE HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)


history_df.to_csv(
    HISTORY_PATH,
    sep="\t",
    index=False
)


print()
print("=" * 92)
print("TRAINING COMPLETATO")
print("=" * 92)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best Val ROC-AUC:",
    f"{best_val_auc:.6f}"
)

print(
    "Checkpoint:",
    BEST_CHECKPOINT_PATH
)

print(
    "History:",
    HISTORY_PATH
)

EUCLIDEAN V3 — TUMOR + HEALTHY — FULL TRAINING
Epoch 01/50 | train loss 0.3809 | train AUC 0.6186 | val loss 0.3903 | val AUC 0.6023 | d+ 0.5491 | d- 0.6271 | gap 0.0781 | d' 0.3659 | 18.6s *BEST*
Epoch 02/50 | train loss 0.3699 | train AUC 0.6391 | val loss 0.3715 | val AUC 0.6439 | d+ 0.5949 | d- 0.7087 | gap 0.1138 | d' 0.5190 | 18.2s *BEST*
Epoch 03/50 | train loss 0.3669 | train AUC 0.6460 | val loss 0.3702 | val AUC 0.6398 | d+ 0.5859 | d- 0.6858 | gap 0.0999 | d' 0.5035 | 18.7s
Epoch 04/50 | train loss 0.3652 | train AUC 0.6492 | val loss 0.3727 | val AUC 0.6416 | d+ 0.5070 | d- 0.5972 | gap 0.0902 | d' 0.5113 | 104.3s
Epoch 05/50 | train loss 0.3614 | train AUC 0.6582 | val loss 0.3684 | val AUC 0.6433 | d+ 0.5712 | d- 0.6712 | gap 0.1001 | d' 0.5146 | 43.9s
Epoch 06/50 | train loss 0.3603 | train AUC 0.6608 | val loss 0.3686 | val AUC 0.6436 | d+ 0.5393 | d- 0.6331 | gap 0.0938 | d' 0.5174 | 18.6s
Epoch 07/50 | train loss 0.3583 | train AUC 0.6659 | val loss 0.3632 | val AUC 0

In [19]:
# ============================================================
# CELL 19 — BEST CHECKPOINT SUMMARY
# ============================================================

best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE
)


model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ],
    strict=True
)


best_pairwise_metrics = evaluate_pairwise(

    model=model,

    loader=val_pair_loader,

    criterion=criterion
)


print("=" * 70)
print("BEST EUCLIDEAN V3 — TUMOR + HEALTHY")
print("=" * 70)

print(
    "Best epoch:",
    best_checkpoint["best_epoch"]
)

print(
    "Val ROC-AUC:",
    f"{best_pairwise_metrics['auc']:.6f}"
)

print(
    "Val loss:",
    f"{best_pairwise_metrics['loss']:.6f}"
)

print(
    "d+:",
    f"{best_pairwise_metrics['d_pos']:.6f}"
)

print(
    "d-:",
    f"{best_pairwise_metrics['d_neg']:.6f}"
)

print(
    "Gap:",
    f"{best_pairwise_metrics['gap']:.6f}"
)

print(
    "d-prime:",
    f"{best_pairwise_metrics['d_prime']:.6f}"
)

BEST EUCLIDEAN V3 — TUMOR + HEALTHY
Best epoch: 19
Val ROC-AUC: 0.661052
Val loss: 0.361724
d+: 0.572445
d-: 0.696728
Gap: 0.124283
d-prime: 0.587036


In [20]:
# ============================================================
# CELL 20 — SINGLE SAMPLE DATASET
# ============================================================

class SingleSampleDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap

        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )

        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


    def __len__(self):

        return len(
            self.metadata
        )


    def __getitem__(
        self,
        index
    ):

        row = int(
            self.rows[index]
        )

        x = np.array(
            self.fcgr_memmap[row],
            dtype=np.float32,
            copy=True
        )

        return {

            "x":
                torch.from_numpy(
                    x
                ).unsqueeze(0),

            "class_id":
                torch.tensor(
                    self.labels[index],
                    dtype=torch.long
                )
        }


# ============================================================
# FULL TRAIN REFERENCE SET
# ============================================================

reference_metadata = (
    metadata[
        metadata["split_cluster"]
        ==
        "train"
    ]
    .copy()
    .reset_index(drop=True)
)


reference_dataset = SingleSampleDataset(

    metadata=
        reference_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


val_single_dataset = SingleSampleDataset(

    metadata=
        val_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


EVAL_BATCH_SIZE = 256


reference_loader = DataLoader(

    reference_dataset,

    batch_size=
        EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


val_single_loader = DataLoader(

    val_single_dataset,

    batch_size=
        EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print(
    "Reference train:",
    len(reference_dataset)
)

print(
    "Validation:",
    len(val_single_dataset)
)

print(
    "Classi reference:",
    len(
        np.unique(
            reference_dataset.labels
        )
    )
)

Reference train: 96167
Validation: 9753
Classi reference: 12


In [21]:
# ============================================================
# CELL 21 — EXTRACT SINGLE-SAMPLE EMBEDDINGS
# ============================================================

def extract_embeddings(
    model,
    loader
):

    model.eval()

    embeddings_list = []
    labels_list = []


    with torch.no_grad():

        for batch in loader:

            x = (
                batch["x"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(

                device_type=
                    DEVICE.type,

                dtype=
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16,

                enabled=
                    DEVICE.type == "cuda"

            ):

                # Usiamo direttamente l'encoder
                z = model.encoder(
                    x
                )


            embeddings_list.append(

                z.float()
                .cpu()
                .numpy()
            )


            labels_list.append(

                batch[
                    "class_id"
                ]
                .numpy()
            )


    return (

        np.concatenate(
            embeddings_list,
            axis=0
        ),

        np.concatenate(
            labels_list,
            axis=0
        ).astype(np.int64)
    )


print(
    "Estrazione train embeddings..."
)


train_embeddings_12, train_labels_12 = (
    extract_embeddings(
        model,
        reference_loader
    )
)


print(
    "Estrazione validation embeddings..."
)


val_embeddings_12, val_labels_12 = (
    extract_embeddings(
        model,
        val_single_loader
    )
)


print(
    "Train:",
    train_embeddings_12.shape
)

print(
    "Val:",
    val_embeddings_12.shape
)

print(
    "Norm media train:",
    np.linalg.norm(
        train_embeddings_12,
        axis=1
    ).mean()
)

Estrazione train embeddings...
Estrazione validation embeddings...
Train: (96167, 128)
Val: (9753, 128)
Norm media train: 1.0


In [23]:
# ============================================================
# CELL 22 — 12-WAY PROTOTYPE CLASSIFICATION
# ============================================================

classes = np.array(
    sorted(
        np.unique(
            train_labels_12
        )
    ),
    dtype=np.int64
)


prototypes = []


for class_id in classes:

    class_embeddings = (
        train_embeddings_12[
            train_labels_12
            ==
            class_id
        ]
    )


    prototype = (
        class_embeddings
        .mean(axis=0)
    )


    prototype = (
        prototype
        /
        (
            np.linalg.norm(
                prototype
            )
            +
            1e-12
        )
    )


    prototypes.append(
        prototype
    )


prototypes = np.stack(
    prototypes,
    axis=0
).astype(np.float32)


# ============================================================
# EUCLIDEAN DISTANCE TO PROTOTYPES
# ============================================================

distances = np.sqrt(

    (
        (
            val_embeddings_12[
                :,
                None,
                :
            ]
            -
            prototypes[
                None,
                :,
                :
            ]
        )
        ** 2
    )
    .sum(axis=2)
)


prediction_indices = (
    distances.argmin(
        axis=1
    )
)


y_pred_12 = (
    classes[
        prediction_indices
    ]
)


prototype_metrics_12 = {

    "accuracy":
        accuracy_score(
            val_labels_12,
            y_pred_12
        ),

    "macro_f1":
        f1_score(
            val_labels_12,
            y_pred_12,
            average="macro",
            zero_division=0
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            val_labels_12,
            y_pred_12
        )
}


print("=" * 72)
print("EUCLIDEAN V3 TRAINED DIRECTLY ON TUMOR + HEALTHY")
print("=" * 72)

print(
    "Accuracy:",
    f"{prototype_metrics_12['accuracy']:.4f}"
)

print(
    "Macro-F1:",
    f"{prototype_metrics_12['macro_f1']:.4f}"
)

print(
    "Balanced Accuracy:",
    f"{prototype_metrics_12['balanced_accuracy']:.4f}"
)

EUCLIDEAN V3 TRAINED DIRECTLY ON TUMOR + HEALTHY
Accuracy: 0.2796
Macro-F1: 0.2434
Balanced Accuracy: 0.2953


In [24]:
# ============================================================
# CELL 23 — EXPERIMENT COMPARISON
# ============================================================

comparison_df = pd.DataFrame(
    [
        {
            "experiment":
                "Euclidean V3 trained 18-way → evaluated 12-way",

            "accuracy":
                0.2808,

            "macro_f1":
                0.2481,

            "balanced_accuracy":
                0.3025
        },

        {
            "experiment":
                "Frozen embedding linear probe 12-way",

            "accuracy":
                0.3071,

            "macro_f1":
                0.2796,

            "balanced_accuracy":
                0.3223
        },

        {
            "experiment":
                "Euclidean V3 trained directly 12-way",

            "accuracy":
                prototype_metrics_12[
                    "accuracy"
                ],

            "macro_f1":
                prototype_metrics_12[
                    "macro_f1"
                ],

            "balanced_accuracy":
                prototype_metrics_12[
                    "balanced_accuracy"
                ]
        }
    ]
)


display(
    comparison_df
)

,experiment,accuracy,macro_f1,balanced_accuracy
0,Euclidean V3 trained 18-way → evaluated 12-way,0.280800,0.248100,0.302500
1,Frozen embedding linear probe 12-way,0.307100,0.279600,0.322300
2,Euclidean V3 trained directly 12-way,0.279606,0.243436,0.295295
